#### Importing Required Libraries

In [2]:
# Importing libraries for data handling.

import pandas as pd
import numpy as np


# Importing libraries for data visualization.

import plotly.express as px
import plotly.graph_objects as go


# Importing libraries for data preprocessing and model development.

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Importing regression models for demand prediction.

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)


# Importing libraries for model evaluation.

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#### Loading the Engineered Dataset

In [3]:
# Loading the engineered dataset for model development.

df = pd.read_csv(
    '../data/processed/engineered_data.csv'
)

In [4]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN


In [5]:
# Checking the dataset structure and data types.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         76000 non-null  object 
 1   Store ID                     76000 non-null  object 
 2   Product ID                   76000 non-null  object 
 3   Category                     76000 non-null  object 
 4   Region                       76000 non-null  object 
 5   Inventory Level              76000 non-null  int64  
 6   Units Sold                   76000 non-null  int64  
 7   Units Ordered                76000 non-null  int64  
 8   Price                        76000 non-null  float64
 9   Discount                     76000 non-null  int64  
 10  Weather Condition            76000 non-null  object 
 11  Promotion                    76000 non-null  int64  
 12  Competitor Pricing           76000 non-null  float64
 13  Seasonality     

##### Converting the Date Column

In [6]:
# Converting the Date column into datetime format.

df['Date'] = pd.to_datetime(
    df['Date']
)

#### Sorting the Dataset by Time

In [9]:
# Sorting the dataset by date to maintain chronological order.

df = df.sort_values(
    'Date'
).reset_index(
    drop=True
)

#### Checking Missing Values

In [10]:
# Checking missing values before preparing the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Rolling_7_Day_Sales            700
Rolling_7_Day_Demand           700
Previous_Units_Sold            100
Previous_Demand                100
Date                             0
Store ID                         0
Product ID                       0
Units Ordered                    0
Price                            0
Discount                         0
Weather Condition                0
Category                         0
Region                           0
Inventory Level                  0
Units Sold                       0
Epidemic                         0
Seasonality                      0
Competitor Pricing               0
Promotion                        0
Demand                           0
Year                             0
Month                            0
Week                             0
Inventory_to_Sales_Ratio         0
Is_Weekend                       0
Day_of_Week                      0
Day                              0
Promotion_Discount               0
Price_Difference_Per

#### Handling the missing values

In [11]:
# Removing rows where lag and rolling features cannot be calculated due to insufficient historical data.

df = df.dropna(
    subset=[
        'Previous_Demand',
        'Previous_Units_Sold',
        'Rolling_7_Day_Demand',
        'Rolling_7_Day_Sales'
    ]
).reset_index(
    drop=True
)

In [12]:
# Verifying that missing values have been removed from the modeling dataset.

df.isnull().sum().sort_values(
    ascending=False
)

Date                           0
Store ID                       0
Product ID                     0
Category                       0
Region                         0
Inventory Level                0
Units Sold                     0
Units Ordered                  0
Price                          0
Discount                       0
Weather Condition              0
Promotion                      0
Competitor Pricing             0
Seasonality                    0
Epidemic                       0
Demand                         0
Year                           0
Month                          0
Week                           0
Day                            0
Day_of_Week                    0
Is_Weekend                     0
Inventory_to_Sales_Ratio       0
Inventory_Gap                  0
Price_Difference               0
Price_Difference_Percentage    0
Promotion_Discount             0
Previous_Demand                0
Previous_Units_Sold            0
Rolling_7_Day_Demand           0
Rolling_7_

In [13]:
# Checking the dataset shape after removing rows with insufficient historical data.

print("Dataset Shape:", df.shape)

Dataset Shape: (75300, 31)


#### Defining Target and Features

In [14]:
# Defining Demand as the target variable for the forecasting models.
target = 'Demand'

# Defining the features that will be used to predict demand.
feature_columns = [
    'Store ID',
    'Product ID',
    'Category',
    'Region',
    'Inventory Level',
    'Units Sold',
    'Units Ordered',
    'Price',
    'Discount',
    'Weather Condition',
    'Promotion',
    'Competitor Pricing',
    'Seasonality',
    'Epidemic',
    'Year',
    'Month',
    'Week',
    'Inventory_to_Sales_Ratio',
    'Is_Weekend',
    'Day_of_Week',
    'Day',
    'Promotion_Discount',
    'Price_Difference_Percentage',
    'Price_Difference',
    'Inventory_Gap',
    'Previous_Demand',
    'Previous_Units_Sold',
    'Rolling_7_Day_Demand',
    'Rolling_7_Day_Sales'
]

# Creating the feature matrix and target variable for model development.

X = df[feature_columns]
y = df[target]

# Verifying the dimensions of the feature matrix and target variable.

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (75300, 29)
Target Shape: (75300,)
